# Fine-Tuning XLM-R: Intención (6 clases, ES + PT) — Kaggle

**Kaggle Notebook** — usa GPU (P100 / T4 x2).

### Setup previo (1 vez):
1. Andá a [kaggle.com/datasets](https://kaggle.com/datasets) → **New Dataset**
2. Subí estos archivos:
   - `bitext_intent_es.csv`
   - `bitext_intent_pt.csv`
3. Ponele nombre: `bitext-intent-es-pt`
4. En el notebook, activá **Accelerator → GPU (P100)** en Settings
5. Agregá el dataset en **Add Data** → buscá `bitext-intent-es-pt`

El modelo final queda en `/kaggle/working/` para descargar.

In [ ]:
# ============================================================
# ENTRENAMIENTO COMPLETO - 6 CLASES + ANTI-LEAK + ANTI-OVERFITTING
# KAGGLE VERSION
# ============================================================

print("="*60)
print("🚀 INICIANDO ENTRENAMIENTO DE INTENCIONES (6 CLASES)")
print("="*60)

# --- 1. VERIFICAR GPU ---
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# --- 2. INSTALAR DEPENDENCIAS ---
!pip install -q datasets accelerate pyarrow tqdm

In [ ]:
# --- 3. IMPORTAR LIBRERÍAS ---
import pandas as pd
import numpy as np
import torch
import os
import json
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset

In [ ]:
# --- 4. CONFIGURACIÓN ---
# En Kaggle los datasets subidos aparecen en /kaggle/input/<nombre-dataset>/
DATA_DIR = '/kaggle/input/datasets/rosaiselagonzalez/bitext-intent-es-pt'
OUTPUT_DIR = '/kaggle/working/xlmr-intent-6clases'

WEIGHT_DECAY = 0.1
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
MAX_LENGTH = 128

print("📋 CONFIGURACIÓN:")
print(f"   weight_decay: {WEIGHT_DECAY}")
print(f"   epochs: {NUM_EPOCHS}")
print(f"   batch_size: {BATCH_SIZE}")
print(f"   learning_rate: {LEARNING_RATE}")
print(f"   data: {DATA_DIR}")

In [ ]:
# --- 5. CARGAR DATOS TRADUCIDOS ---
print("\n📂 1. Cargando datos traducidos...")

es_path = f'{DATA_DIR}/bitext_intent_es.csv'
pt_path = f'{DATA_DIR}/bitext_intent_pt.csv'

print(f"   Buscando en: {es_path}")
print(f"   Buscando en: {pt_path}")

df_es = pd.read_csv(es_path)
df_pt = pd.read_csv(pt_path)

df_es['lang'] = 'es'
df_pt['lang'] = 'pt'
df = pd.concat([df_es, df_pt], ignore_index=True)

print(f"   ✅ Total muestras: {len(df):,}")
print(f"   Columnas: {list(df.columns)}")

In [ ]:
# --- 6. MAPEAR A 6 CLASES ---
print("\n🎯 2. Mapeando 26 intents originales → 6 clases...")

INTENT_MAPPING = {
    'cancel_order': 'cancelacion',
    'check_cancellation_fee': 'cancelacion',
    'complaint': 'reclamo',
    'get_refund': 'reclamo',
    'track_refund': 'reclamo',
    'check_invoice': 'facturacion',
    'get_invoice': 'facturacion',
    'check_payment_methods': 'facturacion',
    'payment_issue': 'facturacion',
    'check_refund_policy': 'facturacion',
    'recover_password': 'problema_tecnico',
    'registration_problems': 'problema_tecnico',
    'review': 'resuelto',
    'contact_customer_service': 'consulta',
    'contact_human_agent': 'consulta',
    'delivery_options': 'consulta',
    'delivery_period': 'consulta',
    'change_order': 'consulta',
    'place_order': 'consulta',
    'track_order': 'consulta',
    'change_shipping_address': 'consulta',
    'set_up_shipping_address': 'consulta',
    'create_account': 'consulta',
    'delete_account': 'consulta',
    'edit_account': 'consulta',
    'switch_account': 'consulta',
    'newsletter_subscription': 'consulta'
}

df['intent'] = df['intent'].map(INTENT_MAPPING)
print(f"   ✅ Mapping: {len(INTENT_MAPPING)} intents → {len(set(INTENT_MAPPING.values()))} clases")

all_intents = sorted(df['intent'].unique())
intent_map = {intent: i for i, intent in enumerate(all_intents)}
id2label = {i: intent for intent, i in intent_map.items()}
df['label'] = df['intent'].map(intent_map)

print(f"   Clases finales: {all_intents}")
print(f"\n   Distribución por clase:")
print(df.groupby(['lang', 'intent']).size().unstack(fill_value=0).to_string())

In [ ]:
# --- 7. SPLIT POR TEXTO ÚNICO (ANTI-LEAK) ---
print("\n🔀 3. Realizando split anti-leak...")

unique_texts = df.groupby('text').agg({
    'intent': 'first',
    'label': 'first',
    'lang': 'first'
}).reset_index()

print(f"   Textos únicos: {len(unique_texts):,} (de {len(df):,} total)")

train_texts, temp_texts = train_test_split(
    unique_texts,
    test_size=0.3,
    random_state=42,
    stratify=unique_texts['intent']
)

val_texts, test_texts = train_test_split(
    temp_texts,
    test_size=0.5,
    random_state=42,
    stratify=temp_texts['intent']
)

train_df = df[df['text'].isin(train_texts['text'])].copy()
val_df = df[df['text'].isin(val_texts['text'])].copy()
test_df = df[df['text'].isin(test_texts['text'])].copy()

print(f"   Train: {len(train_df):,} muestras ({len(train_texts):,} únicos)")
print(f"   Val:   {len(val_df):,} muestras ({len(val_texts):,} únicos)")
print(f"   Test:  {len(test_df):,} muestras ({len(test_texts):,} únicos)")

train_set = set(train_df['text'].unique())
val_set = set(val_df['text'].unique())
test_set = set(test_df['text'].unique())

if len(train_set.intersection(val_set)) == 0 and len(train_set.intersection(test_set)) == 0:
    print("   ✅ No hay leaks entre conjuntos")
else:
    print("   ⚠️ Se detectaron leaks!")

In [ ]:
# --- 8. TOKENIZACIÓN ---
print("\n🔧 4. Tokenizando textos...")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tok(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tok, batched=True)
train_ds = train_ds.remove_columns(['text'])
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

val_ds = Dataset.from_pandas(val_df[['text', 'label']]).map(tok, batched=True)
val_ds = val_ds.remove_columns(['text'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tok, batched=True)
test_ds = test_ds.remove_columns(['text'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"   ✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

In [ ]:
# --- 9. CARGAR MODELO ---
print("\n📦 5. Cargando modelo XLM-RoBERTa...")

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(all_intents),
    id2label=id2label,
    label2id=intent_map,
    ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {'accuracy': (preds == labels).mean()}

# --- PESOS DE CLASE (balanceo) ---
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights_np = compute_class_weight(
    'balanced', classes=all_intents, y=train_df['intent']
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float)
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

print("   📊 Pesos de clase:", {c: f"{w:.2f}" for c, w in zip(all_intents, class_weights_np)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

print(f"   ✅ Modelo cargado: {model.config.num_labels} clases")
print(f"   Map: {id2label}")

In [ ]:
# --- 10. CONFIGURAR ENTRENAMIENTO ---
print("\n⚙️ 6. Configurando entrenamiento...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
    dataloader_num_workers=2,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    max_grad_norm=1.0,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# --- 11. ENTRENAR ---
print("\n" + "="*60)
print("🚀 INICIANDO ENTRENAMIENTO")
print("="*60)
print(f"   weight_decay: {WEIGHT_DECAY}")
print(f"   epochs: {NUM_EPOCHS}")
print(f"   clases: {len(all_intents)}")
print(f"   Train: {len(train_ds):,} muestras")
print(f"   Val: {len(val_ds):,} muestras")
print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("="*60 + "\n")

trainer.train()

In [ ]:
# --- 12. EVALUAR ---
print("\n📊 7. Evaluando modelo...")

results = trainer.predict(test_ds)
preds = np.argmax(results.predictions, axis=1)

test_acc = accuracy_score(test_df['label'].values, preds)

print("\n" + "="*60)
print("📈 RESULTADOS FINALES")
print("="*60)
print(f"🎯 Test Accuracy: {test_acc*100:.2f}%")
print("="*60)

print("\n📋 Reporte por clase:")
print(classification_report(
    test_df['label'].values,
    preds,
    target_names=all_intents,
    digits=4,
    zero_division=0
))

print("\n📊 Accuracy por idioma:")
for lang in ['es', 'pt']:
    mask = test_df['lang'] == lang
    acc = accuracy_score(test_df[mask]['label'].values, preds[mask.values])
    print(f"   {lang.upper()}: {acc*100:.2f}% ({mask.sum():,} muestras)")

In [ ]:
# --- 13. GUARDAR MODELO ---
print("\n💾 8. Guardando modelo...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

mapping_info = {
    'intent_mapping': INTENT_MAPPING,
    'id2label': id2label,
    'label2id': intent_map,
    'classes': all_intents,
    'config': {
        'weight_decay': WEIGHT_DECAY,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE
    }
}

with open(os.path.join(OUTPUT_DIR, 'intent_mapping.json'), 'w') as f:
    json.dump(mapping_info, f, indent=2, ensure_ascii=False)

print(f"✅ Modelo guardado en: {OUTPUT_DIR}")

In [ ]:
# --- 14. SUBIR MODELO A HUGGINGFACE HUB ---
# Kaggle no permite montar Drive, así que subimos a HuggingFace como privado
# (también queda como copia de respaldo permanente)
print("\n🤗 9. Subiendo modelo a HuggingFace Hub...")

from huggingface_hub import HfApi, login
import getpass

HF_TOKEN = getpass.getpass("Token de HuggingFace (Write): ")
login(token=HF_TOKEN)

HF_USER = "rosaiselagonzalez"
REPO_NAME = "xlmr-intent-6clases"
repo_id = f"{HF_USER}/{REPO_NAME}"

api = HfApi()
api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=repo_id,
    path_in_repo=".",
    repo_type="model",
)

print(f"✅ Modelo subido a: https://huggingface.co/{repo_id}")
print(f"\n📥 Después lo descargás local con:")
print(f"   huggingface-cli download {repo_id} --local-dir data/models/intent_xlmr/")

---
## Resumen

| Concepto | Detalle |
|----------|---------|
| Modelo | XLM-RoBERTa Base |
| GPU | T4 x2 (Kaggle) |
| Datos | bitext traducido (27k ES + 27k PT) |
| Total | ~54k muestras reales traducidas |
| Clases | 6 (cancelacion, reclamo, facturacion, problema_tecnico, resuelto, consulta) |
| Anti-leak | Split por texto único |
| Anti-overfitting | weight_decay=0.1, epochs=3, early stopping |
| Alojamiento | HuggingFace Hub (privado) |

**Cero sintéticos.**

Después de correrlo, descargalo local con:
```
huggingface-cli download Rosela/xlmr-intent-6clases --local-dir data/models/intent_xlmr/
```